In [1]:
import pandas as pd
import numpy as np
from shared import load_data, evaluate_model, evaluate_rating_metrics

In [2]:
train_df, test_df, restaurants_df = load_data()
test_users = test_df['user_id'].unique().tolist()
all_restaurants = restaurants_df['business_id'].unique().tolist()

In [3]:
global_mean = train_df['stars'].mean()

user_means = train_df.groupby('user_id')['stars'].mean()
user_bias = user_means - global_mean
user_bias_dict = user_bias.to_dict()

item_means = train_df.groupby('business_id')['stars'].mean()
item_bias = item_means - global_mean
item_bias_dict = item_bias.to_dict()

print(f"Global Mean: {global_mean:.4f}")

Global Mean: 3.9770


In [7]:
def predict_rating(user, item):
    b_u = user_bias_dict.get(user, 0.0)
    b_i = item_bias_dict.get(item, 0.0)

    pred = global_mean+ b_u + b_i

    return np.clip(pred, 1.0, 5.0)

test_df['predicted_stars'] = test_df.apply(lambda row: predict_rating(row['user_id'], row['business_id']), axis=1)

rating_metrics = evaluate_rating_metrics(test_df['stars'], test_df['predicted_stars'])
print("Rating Prediction Performance:")
print(f"MAE: {rating_metrics['MAE']:.4f}")
print(f"RMSE: {rating_metrics['RMSE']:.4f}")

Rating Prediction Performance:
MAE: 0.8636
RMSE: 1.1764


In [6]:
predictions = {}

for user in test_users:
    b_u = user_bias_dict.get(user, 0.0)

    user_item_scores = []
    for item in all_restaurants:
        b_i = item_bias_dict.get(item,0.0)
        score = global_mean + b_u +b_i
        user_item_scores.append((item, score))

    user_item_scores.sort(key=lambda x: x[1], reverse=True)

    top_10_items = [item for item, score in user_item_scores[:10]]
    predictions[user] = top_10_items

ranking_metrics = evaluate_model(predictions, test_df, k=10)
print("\nRanking Performance:")
print(f"Hit Rate @ 10: {ranking_metrics['Hit@10']:.4f}")
print(f"NDCG @ 10: {ranking_metrics['NDCG@10']:.4f}")


Ranking Performance:
Hit Rate @ 10: 0.0033
NDCG @ 10: 0.0013
